In [2]:
from pathlib import Path
import pandas as pd
import re
import numpy as np

IN_DIR  = Path("../data/preprocessed")
OUT_CSV = Path("../data/unified/preprocessed_merged.csv") 

IN_DIR.mkdir(parents=True, exist_ok=True)
OUT_CSV.parent.mkdir(parents=True, exist_ok=True)

# 1. CẤU HÌNH TỪ KHÓA VÀ HÀM TRÍCH XUẤT
hcm_keywords = ['Quận 1', 'Quận 2', 'Quận 3', 'Quận 4', 'Quận 5', 'Quận 6', 'Quận 7', 'Quận 8', 'Quận 9', 
                'Quận 10', 'Quận 11', 'Quận 12', 'Bình Thạnh', 'Gò Vấp', 'Phú Nhuận', 'Bình Tân', 
                'Tân Bình', 'Tân Phú', 'Thủ Đức', 'Bình Chánh', 'Hóc Môn', 'Củ Chi', 'Nhà Bè']

hn_keywords = ['Ba Đình', 'Hoàn Kiếm', 'Tây Hồ', 'Long Biên', 'Cầu Giấy', 'Đống Đa', 'Hai Bà Trưng', 
               'Hoàng Mai', 'Thanh Xuân', 'Hà Đông', 'Bắc Từ Liêm', 'Nam Từ Liêm']

all_keywords = hcm_keywords + hn_keywords

# Tạo dictionary để mapping chính xác Quận -> Thành phố
district_to_city_map = {}
for q in hcm_keywords:
    district_to_city_map[q if 'Quận' in q or 'Huyện' in q else f"Quận {q}"] = 'Hồ Chí Minh'
for q in hn_keywords:
    district_to_city_map[q if 'Quận' in q or 'Huyện' in q else f"Quận {q}"] = 'Hà Nội'

def get_city_from_district(district_name):
    if pd.isna(district_name): return np.nan
    # Trả về thành phố tương ứng từ map, nếu không có trả về NaN
    return district_to_city_map.get(district_name, np.nan)

def final_extract_quan(addr):
    if pd.isna(addr) or addr == "": return np.nan
    s = str(addr)
    m = re.search(r'(Quận|Q\.|Huyện|H\.)\s*([^,]+)', s, flags=re.IGNORECASE)
    if m:
        prefix = m.group(1).strip().capitalize()
        name = m.group(2).strip()
        name = re.sub(r'\(.*\)', '', name).strip()
        if prefix.startswith('Q'): prefix = 'Quận'
        if prefix.startswith('H'): prefix = 'Huyện'
        return f"{prefix} {name}"
    for kw in all_keywords:
        if kw.lower() in s.lower():
            if kw.isdigit() or kw in [str(i) for i in range(1, 13)]:
                return f"Quận {kw}"
            return kw if ('Quận' in kw or 'Huyện' in kw) else f"Quận {kw}"
    return np.nan

def normalize_quan_name(quan_raw):
    if pd.isna(quan_raw): return np.nan
    s = str(quan_raw).strip()
    redundant_suffixes = ['Hà Nội', 'Hồ Chí Minh', 'TP.HCM', 'TP HCM', 'HN', 'HCM']
    for suffix in redundant_suffixes:
        # Thêm dấu ,? để xóa cả trường hợp "Bắc Từ Liêm, Hà Nội"
        s = re.sub(rf'[,\s]+{suffix}$', '', s, flags=re.IGNORECASE).strip()
    return s

def extract_thanh_pho(addr):
    if pd.isna(addr) or addr == "": return np.nan
    s = str(addr).lower()
    if any(p in s for p in ['hồ chí minh', 'tp.hcm', 'tphcm', 'hcm']): return 'Hồ Chí Minh'
    if any(p in s for p in ['hà nội', 'hn']): return 'Hà Nội'
    return np.nan

def infer_city_from_district(district_name):
    if pd.isna(district_name): return np.nan
    name_clean = district_name.replace('Quận ', '').replace('Huyện ', '').lower()
    hcm_list_low = [k.replace('Quận ', '').lower() for k in hcm_keywords]
    hn_list_low = [k.replace('Quận ', '').lower() for k in hn_keywords]
    if any(k in name_clean for k in hcm_list_low): return 'Hồ Chí Minh'
    if any(k in name_clean for k in hn_list_low): return 'Hà Nội'
    return np.nan

def clean_and_dedup_columns(df: pd.DataFrame) -> pd.DataFrame:
    new_cols = []
    keep_idx = []
    seen = set()
    for idx, c in enumerate(df.columns):
        col = str(c).replace("\ufeff", "")
        col = re.sub(r"\s+", " ", col).strip()
        key = col.lower()
        if key not in seen:
            seen.add(key)
            new_cols.append(col)
            keep_idx.append(idx)
    df = df.iloc[:, keep_idx]
    df.columns = new_cols
    return df

def read_relaxed_csv(p: Path) -> pd.DataFrame:
    return pd.read_csv(p, dtype=str, keep_default_na=False, encoding="utf-8-sig", encoding_errors="ignore", on_bad_lines="skip")

paths = sorted(IN_DIR.glob("*.csv"))
if not paths:
    print(f"Không tìm thấy CSV trong {IN_DIR.resolve()}")
else:
    df0 = read_relaxed_csv(paths[0])
    df0 = clean_and_dedup_columns(df0)
    base_cols = list(df0.columns)
    frames = [df0[base_cols]]
    print(f"Chuẩn cột dựa trên: {paths[0].name} -> {base_cols}")

    for p in paths[1:]:
        dfi = read_relaxed_csv(p)
        dfi = clean_and_dedup_columns(dfi)
        for c in base_cols:
            if c not in dfi.columns:
                dfi[c] = ""
        frames.append(dfi[base_cols])
        print(f"Đã gộp: {p.name} (rows={len(dfi)})")

    # BƯỚC GỘP
    merged = pd.concat(frames, ignore_index=True)

    # 2. THỰC HIỆN TRÍCH XUẤT VÀ SUY LUẬN VỊ TRÍ
    print("Đang trích xuất Quận và Thành phố...")
    merged['quan'] = merged['dia_chi'].apply(final_extract_quan)

    merged['quan'] = merged['quan'].apply(normalize_quan_name)

    # Chuẩn hóa cuối cùng cho cột quan
    merged['quan'] = merged['quan'].str.replace('Quận Quận', 'Quận').str.strip()

    # Suy luận Thành phố dựa TRỰC TIẾP trên Quận (Ưu tiên số 1)
    merged['thanh_pho'] = merged['quan'].apply(get_city_from_district)

    # Bước C: Nếu Bước B không ra kết quả (Quận lạ), mới tìm từ khóa trong địa chỉ (Ưu tiên số 2)
    mask_no_city = merged['thanh_pho'].isna()
    merged.loc[mask_no_city, 'thanh_pho'] = merged.loc[mask_no_city, 'dia_chi'].apply(extract_thanh_pho)

    # BỎ DÒNG "Quận H" Ở HÀ NỘI (dữ liệu nhiễu)
    merged = merged[~((merged['thanh_pho'] == 'Hà Nội') & (merged['quan'] == 'Quận H'))]
    
    # TIẾP TỤC LOGIC CŨ CỦA BẠN
    merged['gia'] = pd.to_numeric(merged['gia'], errors='coerce')
    merged['dien_tich_dat'] = pd.to_numeric(merged['dien_tich_dat'], errors='coerce')
    
    merged['gia_tren_m2'] = merged.apply(
        lambda row: round(row['gia'] / row['dien_tich_dat'], 2) 
        if pd.notna(row['gia']) and pd.notna(row['dien_tich_dat']) and row['dien_tich_dat'] > 0
        else None,
        axis=1
    )

    def calculate_diem_sinh_loi(row):
        diem = 0
        tieu_de_lower = str(row.get('tieu_de', '')).lower()
        if 'mặt tiền' in tieu_de_lower or 'mat tien' in tieu_de_lower:
            diem += 5
        
        try:
            so_tang = int(float(row.get('so_tang', 0)))
            so_phong_ngu = int(float(row.get('phong_ngu', 0)))
            so_phong_tam = int(float(row.get('phong_tam', 0)))
            
            diem += so_tang * 2
            diem += so_phong_ngu

            if so_tang > so_phong_ngu:
                ti_le = so_phong_tam / so_tang if so_tang > 0 else 0
            else:
                ti_le = so_phong_tam / so_phong_ngu if so_phong_ngu > 0 else 0
            
            if ti_le >= 1:
                diem += ti_le * so_tang
            else:
                diem -= (1 - ti_le) * so_tang
        except:
            pass
        
        return round(diem, 2)
    
    merged['diem_sinh_loi'] = merged.apply(calculate_diem_sinh_loi, axis=1)
    
    print(f"Đã tạo cột 'quan', 'thanh_pho', 'gia_tren_m2' và 'diem_sinh_loi'")
    merged.to_csv(OUT_CSV, index=False, encoding="utf-8-sig")
    print(f"Gộp xong {len(paths)} file → {OUT_CSV} ({len(merged)} dòng)")

Chuẩn cột dựa trên: batdongsan_preprocessed.csv -> ['tieu_de', 'gia', 'dia_chi', 'dien_tich_dat', 'phong_ngu', 'phong_tam', 'so_tang', 'phap_ly', 'ngay_dang']
Đã gộp: mogi_preprocessed.csv (rows=2079)
Đã gộp: muaban_preprocessed.csv (rows=1759)
Đã gộp: thuviennhadat_preprocessed.csv (rows=1535)
Đang trích xuất Quận và Thành phố...
Đã tạo cột 'quan', 'thanh_pho', 'gia_tren_m2' và 'diem_sinh_loi'
Gộp xong 4 file → ..\data\unified\preprocessed_merged.csv (6973 dòng)
